In [12]:
# -------- LLM (GGUF via CTransformers) --------
from langchain_community.llms import CTransformers

# -------- Embeddings --------
from langchain_community.embeddings import HuggingFaceEmbeddings

# -------- Vector Store (Pinecone) --------
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

# -------- Document Loaders --------
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# -------- Text Splitter --------
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -------- Prompt Templates --------
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate

# -------- Retrieval Chain (NEW Modern Way) --------
from langchain_classic.chains import create_retrieval_chain
from dotenv import load_dotenv
import os


AttributeError: partially initialized module 'torch' has no attribute 'autograd' (most likely due to a circular import)

In [29]:
def load_pdf():
    loader = PyPDFLoader("data/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf")
    documents = loader.load()
    return documents

extracted_data = load_pdf()

In [32]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
              chunk_size = 500 , chunk_overlap = 20
    )
    texts = text_splitter.split_documents(extracted_data)
    return texts

text_chunks = text_split(extracted_data)
len(text_chunks)

6972

In [13]:
def load_embedding_model():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

embeddings = load_embedding_model()

NameError: name 'HuggingFaceEmbeddings' is not defined

In [53]:
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key= PINECONE_API_KEY)
vectorstore = PineconeVectorStore.from_documents(documents=text_chunks, embedding=embeddings, index_name="medical-chatbot")

KeyboardInterrupt: 

In [ ]:
query = "What are Allergies?"

docs = vectorstore.similarity_search(query, k=3)


In [ ]:
prompt_template = """
        You are a professional medical assistant.

        Use ONLY the information provided in the context below to answer the question.
        If the answer is not present in the context, say:
        "I could not find sufficient information in the provided medical source."

        ---------------------
        Context:
        {context}
        ---------------------

        Question:
        {question}

        Provide a well-structured answer using the following format:

        🔷 Definition  
        🔷 Causes  
        🔷 Symptoms  
        🔷 Risk Factors  
        🔷 Diagnosis  
        🔷 Treatment Options  
        🔷 When to See a Doctor  

        Keep the explanation clear, medically accurate, and easy to understand.
        Avoid unnecessary repetition.
 """

prompt = PromptTemplate(
     template=prompt_template,
     input_variables=["context", "question"]
)

context = "\n\n".join([doc.page_content for doc in docs])
